# 101 Spark basics

The goal of this lab is to get familiar with Spark programming.

- Scala
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [RDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/RDD.html)
    - [PairRDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/PairRDDFunctions.html)
- Python
    - [Spark programming guide](https://spark.apache.org/docs/3.5.0/rdd-programming-guide.html)
    - [All RDD APIs](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.RDD.html)

Use `Tab` for autocompletion, `Shift+Tab` for documentation.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local") \
    .appName("Local Spark") \
    .config('spark.ui.port', '4040') \
    .getOrCreate()
sc = spark.sparkContext

sc

<SparkContext master=local appName=Local Spark>

## 101-1 Spark warm-up

Load the ```capra``` and ```divinacommedia``` datasets and try the following actions:
- Show their content (```collect```)
- Count their rows (```count```)
- Split phrases into words (```map``` or ```flatMap```; what’s the difference?)
- Check the results (remember: evaluation is lazy)
- Try the ```toDebugString``` function to check the execution plan
    - In PySpark, use ```toDebugString().decode("unicode_escape")```

In [3]:
rddCapra = sc.textFile("../../../../datasets/capra.txt")
rddDC = sc.textFile("../../../../datasets/divinacommedia.txt")

In [4]:
rddCapra.collect()
#rddDC.collect()

['sopra la panca la capra campa', 'sotto la panca la capra crepa']

In [5]:
print(rddCapra.count(), rddDC.count())

2 14753


In [6]:
rddCapra.map(lambda x : x.split(" ")).collect()
rddCapra.flatMap(lambda x:x.split(" ")).collect()

['sopra',
 'la',
 'panca',
 'la',
 'capra',
 'campa',
 'sotto',
 'la',
 'panca',
 'la',
 'capra',
 'crepa']

In [7]:
rddCapra.map(lambda x : x.split(" ")).toDebugString().decode("unicode_escape")

'(1) PythonRDD[8] at RDD at PythonRDD.scala:53 []\n |  ../../../../datasets/capra.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []\n |  ../../../../datasets/capra.txt HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []'

## 101-2 Basic Spark jobs

Implement on Spark the following jobs and test them on both capra and divinacommedia datasets.

- **Word count**: count the number of occurrences of each word
  - Result: (sopra, 1), (la, 4), …
- **Word length count**: count the number of occurrences of words of given lengths
  - Result: (2, 4), (5, 8)
- Count the average length of words given their first letter (i.e., words that begin with "s" have an average length of 5)
  - Result: (s, 5), (l, 2), …
- Return the inverted index of words (i.e., for each word, list the numbers of lines in which they appear)
  - Result: (sopra, (0)), (la, (0, 1)), ...

Also, check how sorting works and try to sort key-value RDDs by descending values.

In [44]:
def wordify(rows):
    return rows.flatMap(lambda x: x.split(" ")).map(lambda x: x.lower()).filter(lambda x: x != "")

rddCapraWords = wordify(rddCapra)
rddDCWords = wordify(rddDC)
rddCapraWords.collect()

['sopra',
 'la',
 'panca',
 'la',
 'capra',
 'campa',
 'sotto',
 'la',
 'panca',
 'la',
 'capra',
 'crepa']

In [9]:
rddDCWords.map(lambda x: (x, 1)).reduceByKey(lambda x,y: x + y).collect()

[('l', 23510),
 ('a', 42636),
 ('d', 15052),
 ('i', 40267),
 ('v', 8215),
 ('n', 26664),
 ('c', 20922),
 ('o', 37856),
 ('m', 12092),
 ('e', 46804),
 ('t', 22898),
 ('g', 7356),
 ('h', 7114),
 ('r', 26063),
 ('f', 5162),
 (':', 1120),
 ('z', 1856),
 ('s', 22661),
 ('p', 11330),
 ('u', 13544),
 ('�', 6116),
 ('.', 3269),
 ('q', 3353),
 ('!', 234),
 ("'", 7711),
 (';', 1623),
 ('b', 2982),
 (',', 8476),
 ('"', 2231),
 ('?', 282),
 ('-', 19),
 ('~', 1),
 ('x', 141),
 ('y', 1),
 ('(', 3),
 (')', 3),
 ('j', 2)]

In [10]:
rddDCWords.map(lambda x: (len(x), 1)).reduceByKey(lambda x,y: x + y).collect()

[(1, 429569)]

In [46]:
'''
Count the average length of words given their first letter (i.e., words that begin with "s" have an average length 
of 5)
'''
from functools import reduce
rddDCWords \
    .map(lambda word: (word[:1], len(word))) \
    .groupByKey() \
    .mapValues(lambda v: reduce(lambda x,y: x + y, list(v)) / len(list(v))).collect()

[('l', 3.7876783849634528),
 ('d', 4.28529945553539),
 ('c', 4.7628874592697805),
 ('a', 3.9546289493019837),
 ('i', 3.411481900452489),
 ('n', 3.729512587038029),
 ('m', 4.565703971119134),
 ('v', 5.298060809193201),
 ('r', 7.038872691933917),
 ('p', 5.133871380988983),
 ('u', 3.8171641791044775),
 ('s', 4.8543489449305195),
 ('o', 4.3238095238095235),
 ('e', 1.854820415879017),
 ('q', 5.50783378746594),
 ('�', 1.0769230769230769),
 ('f', 4.983743967487935),
 ('t', 4.7542118624509575),
 ('b', 5.8014925373134325),
 ('g', 5.681554265297008),
 ("'", 2.856349757113116),
 ('"', 4.9293381037567086),
 (',', 1.0),
 ('h', 2.7914438502673797),
 ('-', 1.3333333333333333),
 ('z', 5.84),
 ('x', 3.684931506849315),
 (':', 1.0),
 (';', 1.0),
 ('!', 2.7),
 ('(', 4.333333333333333),
 ('.', 1.0),
 ('j', 4.5),
 ('?', 3.0)]

In [28]:
'''
Return the inverted index of words (i.e., for each word, list the numbers of lines in which they appear)
  - Result: (sopra, (0)), (la, (0, 1)), ...
'''
rddCapra \
    .map(lambda x: x.lower().split(" ")) \
    .zipWithIndex() \
    .flatMap(lambda x: map(lambda el: (el, x[1]), filter(lambda y: y != "", x[0]))) \
    .groupByKey() \
    .mapValues(lambda v: set(v)).collect()

[('sopra', {0}),
 ('la', {0, 1}),
 ('panca', {0, 1}),
 ('capra', {0, 1}),
 ('campa', {0}),
 ('sotto', {1}),
 ('crepa', {1})]

## 101-3 Extra Spark jobs

Implement the following job.

- Co-occurrence count: count the number of co-occurrences in the text. A co-occurrence is defined as "two distinct words appearing in the same line".
  - In the first line of the *capra* dataset, co-occurrences are:
     - (sopra, la), (sopra, panca), (sopra, capra), (sopra, campa)
     - (la, sopra), (la, panca), (la, capra), (la, campa) 
     - (panca, sopra), (panca, la), (panca, capra), (panca, campa)
     - (capra, sopra), (capra, la), (capra, panca), (capra, campa)
     - (campa, sopra), (campa, la), (campa, panca), (campa, capra)